## MLflow Prompt Registry with Databricks

### Installing Utilities and Libraries

In [ ]:
%pip install --upgrade "mlflow[databricks]>=3.1.0" databricks-sdk==0.77.0 openai

### Restart the Python Environment

In [ ]:
dbutils.library.restartPython()

### Configure MLflow and Unity Catalog

In [ ]:
import mlflow

# Set Databricks as the MLflow tracking backend
mlflow.set_tracking_uri("databricks")

# MLflow Experiment
experiment_name = "/Shared/prompt-registry-lab"

mlflow.set_experiment(experiment_name)

# Get current catalog
catalog_name = spark.sql(
    "SELECT current_catalog()"
).collect()[0][0]

schema_name = "default"

uc_schema = f"{catalog_name}.{schema_name}"

print("MLflow Experiment:", experiment_name)
print("Prompt Registry Location:", uc_schema)

# associate the MLflow experiment with prompt registry location
mlflow.set_experiment_tags({
    "mlflow.promptRegistryLocation": uc_schema
})

### Create your First Prompt

In [ ]:
prompt_name = "restaurant_review_analyzer"

initial_template = """
You are an AI assistant analyzing restaurant customer reviews.

Analyze the following customer review:

{{review}}

Provide:
1. The overall sentiment
2. The main topic discussed
3. A short response to the customer

Write the response using a {{tone}} tone.
"""

prompt = mlflow.genai.register_prompt(
    name=f"{uc_schema}.{prompt_name}",
    template=initial_template,
    commit_message="Initial restaurant review analysis prompt",
    tags={
        "use_case": "customer_review_analysis",
        "task": "sentiment_analysis",
        "version_stage": "development"
    }
)

# set a production alias
mlflow.genai.set_prompt_alias(
    name=f"{uc_schema}.{prompt_name}",
    alias="production",
    version=1
)

print(f"Prompt: {prompt.name}")
print(f"Version: {prompt.version}")

### Create a Review Analysis Function with LLM Call

In [ ]:
from databricks.sdk import WorkspaceClient
import openai

mlflow.openai.autolog()

@mlflow.trace
def analyze_review(prompt, review: str, tone: str):
    # create the LLM client
    llm_client = WorkspaceClient().serving_endpoints.get_open_ai_client()

    # Fill the Prompt Registry template variables
    formatted_prompt = prompt.format(
        review=review,
        tone=tone
    )

    response = llm_client.chat.completions.create(
        model="databricks-claude-sonnet-4-5",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful customer experience assistant."
            },
            {
                "role": "user",
                "content": formatted_prompt
            }
        ]
    )

    return response.choices[0].message.content

### Load and Execute Version 1 of the Prompt

In [ ]:
# Load and use the prompt in your application
prompt_v1 = mlflow.genai.load_prompt(name_or_uri=f"prompts:/{uc_schema}.{prompt_name}@production")

review = """
The pasta was fantastic and our waiter was extremely friendly.
However, we had to wait almost 40 minutes for our food.
"""

result = analyze_review(
    prompt = prompt_v1,
    review=review,
    tone="professional"
)

print(result)


### Create Version 2 of the Prompt

In [ ]:
improved_template = """
You are an expert customer experience analyst.

Analyze the following restaurant review:

{{review}}

Return your analysis using exactly this structure:

Sentiment:
Positive, Negative, or Mixed

Primary Topic:
Identify the primary topic discussed by the customer.

Key Issue:
Identify the most important issue or compliment.

Recommended Action:
Recommend one action the restaurant should take.

Customer Response:
Write a concise response to the customer using a {{tone}} tone.

Do not invent information that is not contained in the review.
"""

In [ ]:
prompt_v2 = mlflow.genai.register_prompt(
    name=f"{uc_schema}.{prompt_name}",
    template=improved_template,
    commit_message="Added structured analysis and recommended action",
    tags={
        "use_case": "customer_review_analysis",
        "task": "sentiment_analysis",
        "version_stage": "improved"
    }
)

# set a production alias
mlflow.genai.set_prompt_alias(
    name=f"{uc_schema}.{prompt_name}",
    alias="development",
    version=2
)

print(f"Prompt: {prompt.name}")
print(f"Version: {prompt.version}")

### Load and Execute Version 2 of the Prompt

In [ ]:
# Load and use the prompt in your application
prompt_v2 = mlflow.genai.load_prompt(name_or_uri=f"prompts:/{uc_schema}.{prompt_name}@development")

review = """
The pasta was fantastic and our waiter was extremely friendly.
However, we had to wait almost 40 minutes for our food.
"""

result = analyze_review(
    prompt = prompt_v2,
    review=review,
    tone="professional"
)

print(result)
